In [16]:
import numpy as np
import pandas as pd

In [17]:
df = pd.read_csv("./data.csv")

In [19]:
categorical = ['lead_source', 'industry', 'employment_status', 'location']
numerical = ['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score', 'converted']

for c in categorical:
    df[c] = df[c].fillna("NA")

for n in numerical:
    df[n] = df[n].fillna(0.0)

df.isnull().sum()

lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64

In [20]:
# Question 1: technology

df.industry.mode()

0    technology
Name: industry, dtype: str

In [ ]:
# Question 2: interaction_count and lead_score

df[numerical].corr()

,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
annual_income,1.000000,0.161300,0.122842,0.229496,0.189652
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204,0.395038
interaction_count,0.122842,0.721609,1.000000,0.915746,0.448624
lead_score,0.229496,0.757204,0.915746,1.000000,0.483246
converted,0.189652,0.395038,0.448624,0.483246,1.000000


In [22]:
from sklearn.model_selection import train_test_split

In [23]:
# Split the data

df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

df_full_train = df_full_train.reset_index(drop=True)
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

y_full_train = df_full_train.converted.values
y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values

del df_full_train['converted']
del df_train['converted']
del df_val['converted']
del df_test['converted']

In [24]:
from sklearn.metrics import mutual_info_score


def mutual_info_churn_score(series):
    return mutual_info_score(series, y_train)

In [26]:
# Question 3: lead_source

mi = df_train[categorical].apply(mutual_info_churn_score)
mi.sort_values(ascending=False)

lead_source          0.034346
employment_status    0.018966
industry             0.001995
location             0.001043
dtype: float64

In [27]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

In [ ]:
# Question 4: 0.65
dv = DictVectorizer(sparse=False)

# same as numerical without converted
n = ['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']
train_dict = df_train[categorical + n].to_dict(orient='records')
X_train = dv.fit_transform(train_dict)

val_dict = df_val[categorical + n].to_dict(orient='records')
X_val = dv.transform(val_dict)

model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict_proba(X_val)[:, 1]
converted_decision = (y_pred >= 0.5)
accuracy = (y_val == converted_decision).mean()
accuracy

np.float64(0.645)

In [34]:
# Question 5: number_of_courses_viewed

for feature in ['lead_source', 'number_of_courses_viewed', 'interaction_count']:
    n = ['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']
    c = ['lead_source', 'industry', 'employment_status', 'location']

    if feature in c:
        c.remove(feature)

    if feature in n:
        n.remove(feature)

    train_dict = df_train[c + n].to_dict(orient='records')
    X_train = dv.fit_transform(train_dict)

    val_dict = df_val[c + n].to_dict(orient='records')
    X_val = dv.transform(val_dict)

    model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    
    y_pred = model.predict_proba(X_val)[:, 1]
    converted_decision = (y_pred >= 0.5)
    acc = (y_val == converted_decision).mean()
    print(f'{feature}: {(accuracy - acc)}')

lead_source: 0.0030000000000000027
number_of_courses_viewed: 0.0020000000000000018
interaction_count: 0.04400000000000004


In [ ]:
# Question 6: 0.001

for r in [0.000001, 0.00001, 0.0001, 0.001]:
    n = ['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score']
    c = ['lead_source', 'industry', 'employment_status', 'location']

    train_dict = df_train[c + n].to_dict(orient='records')
    X_train = dv.fit_transform(train_dict)

    val_dict = df_val[c + n].to_dict(orient='records')
    X_val = dv.transform(val_dict)

    model = LogisticRegression(solver='liblinear', C=r, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    
    y_pred = model.predict_proba(X_val)[:, 1]
    converted_decision = (y_pred >= 0.5)
    acc = (y_val == converted_decision).mean()
    print(f'{r}: {(acc)}')

1e-06: 0.598
1e-05: 0.598
0.0001: 0.613
0.001: 0.645
